# Exercise 1: Image characteristics

In [1]:
import numpy as np
import pylab as pl
import matplotlib.pyplot as plt
from pathlib import Path

import caiman as cm

import seaborn as sns

: 

In [ ]:
input_tif_file_path = '../data/Calcium Imaging data/Ca DATA/caiman_video_trial_0.tif'

original_movie = cm.load(input_tif_file_path)
T, H, W = original_movie.shape
print(f'Frames: {T}, Height: {H}, Width: {W}')

## A. Distinguishing pixels

I select N random pixels from the ROI and plot their temporal fluorescence trace. Pixels that are on  an active neuron show the classic calcium transient dynamic: a fast rise followed by a slow decay.

In [ ]:
np.random.seed(42)
N = 8

rows = np.random.randint(0, H, N)
cols = np.random.randint(0, W, N)

In [ ]:
figure, axes = plt.subplots(N, 1, figsize=(12, 10), sharex=True)

for i, (r, c) in enumerate(zip(rows, cols)):
    axes[i].plot(original_movie[:, r, c], lw=0.8, c='steelblue')
    axes[i].set_ylabel(f'({r},{c})', fontsize=9, rotation=0, labelpad=45)
    sns.despine(ax=axes[i])

axes[-1].set_xlabel('time (frames)')
figure.suptitle('Temporal traces of randomly selected pixels')
plt.tight_layout()

Some pixels display sharp transient events (fast rise, slow decay) indicating that they lie on an active neuron. Others show low-amplitude fluctuations typical of background or neuropil. Re-run the cell above with a different seed to sample new locations.

## B. Temporal statistics of pixel activation

The histogram of pixel values over time reflects how a pixel behaves across the recording. A neuron pixel spends most time near its baseline but occasionally reaches high values during transients, producing a right-skewed distribution. A background pixel stays close to a stable mean, giving a narrower, more symmetric distribution.

In [ ]:
figure, axes = plt.subplots(2, N // 2, figsize=(14, 6))
axes = axes.ravel()

for i, (r, c) in enumerate(zip(rows, cols)):
    trace = np.array(original_movie[:, r, c])
    axes[i].hist(trace, bins=30, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[i].set_title(f'pixel ({r},{c})', fontsize=9)
    axes[i].set_xlabel('fluorescence (a.u.)', fontsize=8)
    sns.despine(ax=axes[i])

figure.suptitle('Pixel value distributions')
plt.tight_layout()

In [ ]:
mean_image = cm.summary_images.mean_image(input_tif_file_path)

figure, axes = plt.subplots(1, 1, figsize=(6, 5))
mesh = axes.pcolormesh(mean_image, cmap='gray')
axes.scatter(cols, rows, s=60, c='red', marker='x', linewidths=1.2, label='sampled pixels')
axes.set_xlabel('x')
axes.set_ylabel('y')
axes.set_title('Mean image with sampled pixel locations')
axes.legend(fontsize=9)
figure.colorbar(mesh, ax=axes)
plt.tight_layout()

**Differences across regions.**  
Pixels on a neuron show a higher mean fluorescence and a right-skewed distribution with a heavy tail: the long baseline periods are punctuated by brief bright events. Background pixels have a narrower, more symmetric distribution concentrated around a stable low value.

**Why not work directly with pixel traces?**  

Even though calcium dynamics are sometimes visible in individual pixels, using raw pixel traces as the unit of analysis has several problems:

1. **Low SNR.** A single pixel integrates photons over a tiny area, so shot noise and read noise dominate. Integrating over the full spatial footprint of a cell body gives a much higher signal-to-noise ratio.

2. **Spatial mixing.** A neuron spans many pixels and those pixels also overlap with neuropil and neighbouring cells. A single pixel trace is a mixture of multiple sources, with no clean one-to-one correspondence to a single neuron.

3. **Redundancy and high dimensionality.** Pixels from the same cell are highly correlated, so treating each pixel as an independent signal inflates the dimensionality without adding information, making downstream analyses (spike inference, population decoding) impractical.

4. **No cell identity.** Raw pixel traces cannot be attributed to individual neurons. Source extraction algorithms such as CNMF solve a blind source-separation problem that jointly estimates spatial footprints and temporal traces, returning one denoised trace per neuron.